<a href="https://colab.research.google.com/github/sanjeevchikkam/Medicinemapping/blob/main/MedicineMapping(Sentence%20Transformers).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Start the project

## Install and import libraries

In [1]:
!pip install rapidfuzz
import pandas as pd
import re
from rapidfuzz import process, fuzz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 15.3 MB/s eta 0:00:00


### upload the pharmacy and master medicine id and name

In [2]:
from google.colab import files
uploaded = files.upload()

Saving pharmacy_medicine.csv to pharmacy_medicine.csv
Saving master_medicine.csv to master_medicine.csv


In [3]:
pharmacy = pd.read_csv("pharmacy_medicine.csv")
master = pd.read_csv("master_medicine.csv")

In [4]:
pharmacy["medicine_name"]

,medicine_name
0,Paracetamol 500mg Tab
1,Paracetamol 650mg Tablet
2,Amoxycillin 250mg Cap
3,Amoxycillin 500 mg Capsule
4,Ibuprofen 200mg Tab
5,Ibuprofen 400mg Tablet
6,Cetrizine 10mg Tab
7,Levocetrizine 5 mg Tablet
8,Metformin 500mg Tab
9,Metformin 1g Tablet


# Write the Normalize text

In [5]:
def normalize_text(text):

    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [6]:
pharmacy["normalized_name"] = pharmacy["medicine_name"].apply(normalize_text)
master["normalized_name"] = master["medicine_name"].apply(normalize_text)

In [7]:
master["normalized_name"]

,normalized_name
0,paracetamol 500 mg tablet
1,paracetamol 650 mg tablet
2,amoxicillin 250 mg capsule
3,amoxicillin 500 mg capsule
4,ibuprofen 200 mg tablet
5,ibuprofen 400 mg tablet
6,cetirizine 10 mg tablet
7,levocetirizine 5 mg tablet
8,metformin 500 mg tablet
9,metformin 1000 mg tablet


In [8]:
pharmacy["normalized_name"]

,normalized_name
0,paracetamol 500mg tab
1,paracetamol 650mg tablet
2,amoxycillin 250mg cap
3,amoxycillin 500 mg capsule
4,ibuprofen 200mg tab
5,ibuprofen 400mg tablet
6,cetrizine 10mg tab
7,levocetrizine 5 mg tablet
8,metformin 500mg tab
9,metformin 1g tablet


### replace tab -> tablet

In [9]:
abbrev = {
    r"\btab\b": "tablet",
    r"\bcap\b": "capsule",
    r"\binj\b": "injection",
    r"\bsyp\b": "syrup"
}

pharmacy["normalized_name"] = pharmacy["normalized_name"].replace(abbrev, regex=True)

In [10]:
abbrev = {
    "tab": "tablet",
    "cap": "capsule",
    "inj": "injection",
    "syp": "syrup"
}

for k, v in abbrev.items():
    pharmacy["normalized_name"] = pharmacy["normalized_name"].str.replace(
        rf"\b{k}\b", v, regex=True
    )

In [11]:
pharmacy["normalized_name"]

,normalized_name
0,paracetamol 500mg tablet
1,paracetamol 650mg tablet
2,amoxycillin 250mg capsule
3,amoxycillin 500 mg capsule
4,ibuprofen 200mg tablet
5,ibuprofen 400mg tablet
6,cetrizine 10mg tablet
7,levocetrizine 5 mg tablet
8,metformin 500mg tablet
9,metformin 1g tablet


In [12]:
master["normalized_name"]

,normalized_name
0,paracetamol 500 mg tablet
1,paracetamol 650 mg tablet
2,amoxicillin 250 mg capsule
3,amoxicillin 500 mg capsule
4,ibuprofen 200 mg tablet
5,ibuprofen 400 mg tablet
6,cetirizine 10 mg tablet
7,levocetirizine 5 mg tablet
8,metformin 500 mg tablet
9,metformin 1000 mg tablet


In [13]:
master_names = master["normalized_name"].tolist()

## Match Master

#Fuzzy match

In [14]:
def match_master(medicine):

    match, score, index = process.extractOne(
        medicine,
        master_names,
        scorer=fuzz.token_sort_ratio
    )

    master_id = master.iloc[index]["master_id"]

    return pd.Series([master_id, match, score])

## calling and matching the pharmacy medi name

In [15]:
pharmacy[["master_id","matched_name","score"]] = pharmacy["normalized_name"].apply(match_master)

In [16]:
final_inventory = pharmacy[[
    "pharmacy_id",
    "medicine_name",
    "master_id",
    "matched_name",
    "score"
]]

final_inventory = final_inventory.rename(columns={
    "medicine_name":"pharmacy_name"
})

print(final_inventory)

   pharmacy_id                 pharmacy_name master_id  \
0         P001         Paracetamol 500mg Tab      M001   
1         P002      Paracetamol 650mg Tablet      M002   
2         P003         Amoxycillin 250mg Cap      M003   
3         P004    Amoxycillin 500 mg Capsule      M004   
4         P005           Ibuprofen 200mg Tab      M005   
5         P006        Ibuprofen 400mg Tablet      M006   
6         P007            Cetrizine 10mg Tab      M007   
7         P008     Levocetrizine 5 mg Tablet      M008   
8         P009           Metformin 500mg Tab      M009   
9         P010           Metformin 1g Tablet      M010   
10        P011             Amlodipin 5mg Tab      M011   
11        P012        Amlodipine 10mg Tablet      M012   
12        P013         Atorvastatin 10mg Tab      M013   
13        P014            Atorva 20mg Tablet      M014   
14        P015        Azithromycin 250mg Tab      M015   
15        P016          Azithro 500mg Tablet      M016   
16        P017

# To Download

In [17]:
final_inventory.to_csv("pharmacy_inventory.csv", index=False)

## Final Inventory csv file

In [18]:
from google.colab import files
files.download("pharmacy_inventory.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Sentence Transformer

In [19]:
!pip install sentence-transformers

In [ ]:
##import senetence transformers and cosine similarity

In [20]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [21]:
model = SentenceTransformer('all-MiniLM-L6-v2')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## Embeddings

In [22]:
pharmacy_embeddings = model.encode(pharmacy["normalized_name"].tolist())
master_embeddings = model.encode(master["normalized_name"].tolist())

In [23]:
similarity_matrix = cosine_similarity(pharmacy_embeddings, master_embeddings)

In [25]:
best_match_index = similarity_matrix.argmax(axis=1)
best_scores = similarity_matrix.max(axis=1)

pharmacy["master_id"] = master.iloc[best_match_index]["master_id"].values
pharmacy["score"] = best_scores

## Result

In [26]:
final_inventory = pharmacy[[
    "pharmacy_id",
    "medicine_name",
    "master_id",
    "score"
]]

final_inventory = final_inventory.rename(columns={
    "medicine_name": "pharmacy_name"
})

In [27]:
final_inventory

,pharmacy_id,pharmacy_name,master_id,score
0,P001,Paracetamol 500mg Tab,M001,0.993024
1,P002,Paracetamol 650mg Tablet,M002,0.992442
2,P003,Amoxycillin 250mg Cap,M003,0.960376
3,P004,Amoxycillin 500 mg Capsule,M004,0.965562
4,P005,Ibuprofen 200mg Tab,M005,0.991051
5,P006,Ibuprofen 400mg Tablet,M006,0.991929
6,P007,Cetrizine 10mg Tab,M007,0.846675
7,P008,Levocetrizine 5 mg Tablet,M008,0.951114
8,P009,Metformin 500mg Tab,M009,0.992130
9,P010,Metformin 1g Tablet,M010,0.883087


In [28]:
final_inventory.to_csv("pharmacy_inventory.csv", index=False)

##Final inventory csv file

In [29]:
from google.colab import files
files.download("pharmacy_inventory.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>